# Set Up Environment


In [4]:
# [Cell 1: Check GPU]
!nvidia-smi

Sat Mar 14 13:04:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               Off |   00000000:4A:00.0 Off |                    0 |
| N/A   42C    P0             99W /  350W |    5719MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
# [Cell 2: Cleanup]
!rm -rf ./sample_data

## Define Directory Path Variables

In [6]:
# [Cell 3: Imports - Core]
import os
import pandas as pd

In [7]:
# [Cell 4: Constants] *** ALWAYS RUN ON RESTART ***
IMAGE_DIR = "./images"
TRAIN_DIR = "./train"
VAL_DIR = "./val"
DATASET_DIR = "./dataset"
ORIGINAL_CSV = "./dataset/Data_Entry_2017.csv"
BINARY_MATRIX_CSV = "all_labels.csv"
TRAIN_CSV = "train_labels.csv"
VAL_CSV = "val_labels.csv"

print("Constants defined: IMAGE_DIR, TRAIN_DIR, VAL_DIR, DATASET_DIR, ORIGINAL_CSV, BINARY_MATRIX_CSV, TRAIN_CSV, VAL_CSV")

Constants defined: IMAGE_DIR, TRAIN_DIR, VAL_DIR, DATASET_DIR, ORIGINAL_CSV, BINARY_MATRIX_CSV, TRAIN_CSV, VAL_CSV


## Download Data from URL (alternative for downloading from Kaggle)

In [ ]:
# [Cell 5: Imports - Download]
import tarfile
import urllib.request

<B> SKIP THE CELLS BELOW if you downloaded images via Kaggle CLI.


In [ ]:
# [Cell 6: Download & Extract Images]
# SKIP if you downloaded images via Kaggle CLI. Uses IMAGE_DIR, TRAIN_DIR, VAL_DIR, DATASET_DIR from Cell 4.

# Check if images already exist (either in ./images/ or already split into ./train/)
images_in_source = len(os.listdir(IMAGE_DIR)) if os.path.exists(IMAGE_DIR) else 0
images_in_train = len(os.listdir(TRAIN_DIR)) if os.path.exists(TRAIN_DIR) else 0

if images_in_source > 1000 or images_in_train > 1000:
    print(f"Images already exist!")
    print(f"  ./images/: {images_in_source} files")
    print(f"  ./train/:  {images_in_train} files")
    print("Skipping download. Delete these folders to re-download.")
else:
    # URLs for the tar.gz files (6 of 12 archives from NIH)
    links = [
        'https://nihcc.box.com/shared/static/vfk49d74nhbxq3nqjg0900w5nvkorp5c.gz',
        'https://nihcc.box.com/shared/static/i28rlmbvmfjbl8p2n3ril0pptcmcu9d1.gz',
        'https://nihcc.box.com/shared/static/f1t00wrtdk94satdfb9olcolqx20z2jp.gz',
        'https://nihcc.box.com/shared/static/0aowwzs5lhjrceb3qp67ahp0rd1l1etg.gz',
        'https://nihcc.box.com/shared/static/v5e3goj22zr6h8tzualxfsqlqaygfbsn.gz',
        'https://nihcc.box.com/shared/static/asi7ikud9jwnkrnkj99jnpfkjdes7l6l.gz',
    ]

    os.makedirs(DATASET_DIR, exist_ok=True)

    for index, link in enumerate(links):
        tar = "images_%02d.tar.gz" % (index + 1)
        tar_path = os.path.join(DATASET_DIR, tar)
        urllib.request.urlretrieve(link, tar_path)
        print(f"Downloaded {tar}")

        with tarfile.open(tar_path, "r") as tar:
            for member in tar.getmembers():
                if member.isfile() and member.name.lower().endswith(".png"):
                    member.name = os.path.basename(member.name)
                    tar.extract(member, path=IMAGE_DIR)
        print(f"Extracted {tar}")

        os.remove(tar_path)

    print("Download and extraction complete")

# Data Preparation

<B>SKIP CELL BELOW if all_labels.csv already exists

In [ ]:
# [Cell 7: Create Binary Label Matrix] *** SKIP if all_labels.csv already exists ***
df = pd.read_csv(ORIGINAL_CSV, usecols=["Image Index", "Finding Labels"])
df["Finding Labels"] = df["Finding Labels"].fillna("")

unique_labels_set = set()
labels_series = df["Finding Labels"].str.split("|")
for labels in labels_series:
    for label in labels:
        if label and label != "No Finding":
            unique_labels_set.add(label)
unique_labels = sorted(unique_labels_set)

for label in unique_labels:
    df[label] = df["Finding Labels"].apply(lambda x: 1 if label in x.split("|") else 0)
binary_df = df[["Image Index"] + unique_labels]

binary_df.to_csv(BINARY_MATRIX_CSV, index=False)
print(f"Created {BINARY_MATRIX_CSV}")

<B> SKIP cell below if train_labels.csv and val_labels.csv already exist 

In [ ]:
# [Cell 8: Train/Val Split] *** SKIP if train_labels.csv and val_labels.csv already exist ***
# Creates train_df and val_df (Create Symlinks loads from CSV directly)
RANDOM_SEED = 42

df = pd.read_csv(BINARY_MATRIX_CSV)
train_df = df.sample(frac=0.8, random_state=RANDOM_SEED)
val_df = df.drop(train_df.index)

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
print(f"Created {TRAIN_CSV} ({len(train_df)} rows) and {VAL_CSV} ({len(val_df)} rows)")

<b>SKIP cell below if ./train and ./val already have symlinks

In [5]:
# [Cell 9: Create Symlinks] *** SKIP if ./train and ./val already have symlinks (~20 min) ***
# Loads CSVs directly - no dependency on Cell 8's in-memory variables

train_images = pd.read_csv(TRAIN_CSV)["Image Index"]
val_images = pd.read_csv(VAL_CSV)["Image Index"]

if os.path.exists(TRAIN_DIR):
    for f in os.listdir(TRAIN_DIR):
        os.remove(os.path.join(TRAIN_DIR, f))
if os.path.exists(VAL_DIR):
    for f in os.listdir(VAL_DIR):
        os.remove(os.path.join(VAL_DIR, f))

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

abs_image_dir = os.path.abspath(IMAGE_DIR)

for image in train_images:
    src = os.path.join(abs_image_dir, image)
    dst = os.path.join(TRAIN_DIR, image)
    if os.path.exists(src):
        os.symlink(src, dst)

for image in val_images:
    src = os.path.join(abs_image_dir, image)
    dst = os.path.join(VAL_DIR, image)
    if os.path.exists(src):
        os.symlink(src, dst)

print(f"Created {len(os.listdir(TRAIN_DIR))} symlinks in {TRAIN_DIR}")
print(f"Created {len(os.listdir(VAL_DIR))} symlinks in {VAL_DIR}")

Created 89696 symlinks in ./train
Created 22424 symlinks in ./val


# Training Pipeline

In [8]:
# [Cell 10: Imports - Training]
import torch
import torch.nn as nn
from torchvision import transforms, models, datasets
import PIL
import time
from torch.cuda.amp import GradScaler, autocast
import mlflow


In [9]:
# [Cell 11: Device & MLflow Setup]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# MLflow: where to store runs (params, metrics, artifacts)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

Using device: cuda
MLflow tracking URI: sqlite:///mlflow.db


In [10]:
# [Cell 12: MLflow Experiment]
mlflow.set_experiment("CXDD-Model")
print(f"MLflow experiment: {mlflow.get_experiment_by_name('CXDD-Model').name}")

2026/03/14 13:06:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/14 13:06:32 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


MLflow experiment: CXDD-Model


In [11]:
# [Cell 13: Training Hyperparameters]
model_name = "ResNet50"  # Change when switching models (e.g., EfficientNet, DenseNet)
batch_size = 128
learning_rate = 2e-4
num_epochs = 12

In [12]:
# [Cell 14: Dataset Class]
class ChestXrayDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, csv_file, transform=None):
        self.images_dir = images_dir
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform

        # Original names and labels
        image_names = self.annotations.iloc[:, 0].values
        labels = self.annotations.iloc[:, 1:].values.astype(float)

        # Keep only rows whose image file exists
        existing_mask = [
            os.path.exists(os.path.join(images_dir, img_name)) for img_name in image_names
        ]
        if not any(existing_mask):
            raise FileNotFoundError(
                f"No images found in {images_dir} that match entries in {csv_file}"
            )

        self.image_names = [name for name, keep in zip(image_names, existing_mask) if keep]
        self.labels = labels[existing_mask]

        self.num_classes = self.labels.shape[1]
        self.class_labels = list(self.annotations.columns)[1:]

        missing_count = len(image_names) - len(self.image_names)
        if missing_count > 0:
            print(
                f"ChestXrayDataset: Skipped {missing_count} missing files in '{images_dir}'. "
                f"Using {len(self.image_names)} images."
            )

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = os.path.join(self.images_dir, self.image_names[idx])
        # Double-check existence in case files changed after init
        if not os.path.exists(img_name):
            raise FileNotFoundError(f"Image no longer exists: {img_name}")
        image = PIL.Image.open(img_name).convert('RGB')
        labels = torch.FloatTensor(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, labels

In [13]:
# [Cell 15: Data Transforms]
# Resize, normalize to ImageNet stats for transfer learning
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [14]:
# [Cell 16: Create DataLoaders]
# Create datasets and dataloaders for training and validation
train_dataset = ChestXrayDataset(
    images_dir=TRAIN_DIR,
    csv_file=TRAIN_CSV,
    transform=transform
)
val_dataset = ChestXrayDataset(
    images_dir=VAL_DIR,
    csv_file=VAL_CSV,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

## Model Architecture & Optimizer

In [19]:
# [Cell 17: Model Definition]
# ResNet50 pretrained on ImageNet, with custom fully-connected layers for 14-class multilabel
num_labels = train_dataset.num_classes
class_labels = train_loader.dataset.class_labels
print(f"Number of labels: {num_labels}")
print(f"Labels: {class_labels}")

model = models.resnet50(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_labels)
)

# Unfreeze fc layers
for param in model.fc.parameters():
    param.requires_grad = True

# Use DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)

model = model.to(device)

Number of labels: 14
Labels: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


/u50/moucattv/CXDD_model/cxdd_env/lib64/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/u50/moucattv/CXDD_model/cxdd_env/lib64/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using 4 GPUs with DataParallel!


In [20]:
# [Cell 18: Loss & Optimizer]
# BCEWithLogitsLoss for multilabel classification, Adam optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


# Training

In [21]:
# [Cell 19: Training Loop]
# Train with early stopping, checkpointing, progress tracking, and mixed precision
patience = 3
counter = 0
best_val_loss = float("inf")

run_name = input("Enter a name for this run: ").strip()
if not run_name:
    run_name = f"{model_name}_lr{learning_rate}_bs{batch_size}_ep{num_epochs}"  # fallback if empty

with mlflow.start_run(run_name=run_name):
    # Log hyperparameters for this run
    mlflow.log_params({
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "patience": patience,
        "optimizer": "Adam",
        "loss": "BCEWithLogitsLoss",
        "model": model_name,
    })
    run_best_model_path = f"best_model_{mlflow.active_run().info.run_id}.pth"
    scaler = GradScaler()
    start_time = time.time()


    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # Mixed precision forward pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # Mixed precision backward pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_running_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)

        val_loss = val_running_loss / len(val_loader.dataset)

        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {epoch_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")

        # MLflow: log metrics per epoch
        mlflow.log_metric("train_loss", epoch_loss, step=epoch+1)
        mlflow.log_metric("val_loss", val_loss, step=epoch+1)

        # Checkpointing: save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save(model_to_save.state_dict(), run_best_model_path)
            print(f"  Saved new best model (val_loss: {val_loss:.4f})")
        else:
            counter += 1
            print(f"  No improvement ({counter}/{patience})")
            if counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        torch.cuda.empty_cache()

    end_time = time.time()

    # MLflow: log best model in MLflow format (MLmodel, requirements.txt, etc.)
    model_to_log = model.module if hasattr(model, 'module') else model
    model_to_log.load_state_dict(torch.load(run_best_model_path, weights_only=True))
    mlflow.pytorch.log_model(model_to_log, artifact_path="model")

    # Store run_id for evaluation cell to log metrics to same run
    mlflow_run_id = mlflow.active_run().info.run_id

    print(f"\nTraining complete in {(end_time - start_time)/60:.1f} minutes")
    print(f"Best validation loss: {best_val_loss:.4f}")

/tmp/ipykernel_1292842/920894137.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1292842/920894137.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1292842/920894137.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/12]
  Train Loss: 0.1934
  Val Loss: 0.1734
  Saved new best model (val_loss: 0.1734)
Epoch [2/12]
  Train Loss: 0.1786
  Val Loss: 0.1722
  Saved new best model (val_loss: 0.1722)
Epoch [3/12]
  Train Loss: 0.1766
  Val Loss: 0.1703
  Saved new best model (val_loss: 0.1703)
Epoch [4/12]
  Train Loss: 0.1751
  Val Loss: 0.1692
  Saved new best model (val_loss: 0.1692)
Epoch [5/12]
  Train Loss: 0.1745
  Val Loss: 0.1683
  Saved new best model (val_loss: 0.1683)
Epoch [6/12]
  Train Loss: 0.1736
  Val Loss: 0.1688
  No improvement (1/3)
Epoch [7/12]
  Train Loss: 0.1729
  Val Loss: 0.1677
  Saved new best model (val_loss: 0.1677)
Epoch [8/12]
  Train Loss: 0.1726
  Val Loss: 0.1675
  Saved new best model (val_loss: 0.1675)
Epoch [9/12]
  Train Loss: 0.1721
  Val Loss: 0.1683
  No improvement (1/3)
Epoch [10/12]
  Train Loss: 0.1715
  Val Loss: 0.1682
  No improvement (2/3)
Epoch [11/12]
  Train Loss: 0.1712
  Val Loss: 0.1666
  Saved new best model (val_loss: 0.1666)
Epoch [12/

2026/03/14 14:05:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 14:06:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Training complete in 49.3 minutes
Best validation loss: 0.1666


In [15]:
# [Cell 20: Imports - Evaluation & Utils]
import numpy as np
from sklearn.metrics import roc_auc_score

# Load & Eval Helper Functions

In [16]:
# [Cell 21: Load Model Function]
def load_model(run_id=None):
    """Load model from MLflow. run_id=None uses latest run (or mlflow_run_id from training). Returns (model, run_id)."""
    if run_id:
        print(f"Loading specified run: {run_id}")
    elif "mlflow_run_id" in globals() and mlflow_run_id:
        run_id = mlflow_run_id
        print(f"Loading most recent training run: {run_id}")
    else:
        runs = mlflow.search_runs(experiment_names=["CXDD-Model"], order_by=["start_time DESC"], max_results=1)
        if runs.empty:
            raise RuntimeError("No runs found in CXDD-Model experiment. Train a model first.")
        run_id = runs.iloc[0]["run_id"]
        print(f"Loading latest run: {run_id}")

    model = mlflow.pytorch.load_model(f"runs:/{run_id}/model")
    model = model.to(device)
    model.eval()
    return model, run_id

In [17]:
# [Cell 22: Evaluation Helper Functions]
from sklearn.metrics import precision_score, recall_score, f1_score

def run_inference(model, dataloader, device, threshold=0.3):
    """Run model inference and return probs, preds, labels."""
    model.eval()
    probs_list, preds_list, labels_list = [], [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = probs > threshold
            probs_list.append(probs.cpu().numpy())
            preds_list.append(preds.cpu().numpy())
            labels_list.append(labels.cpu().numpy())
    return np.concatenate(probs_list), np.concatenate(preds_list), np.concatenate(labels_list)


def compute_per_class_metrics(all_probs, all_preds, all_labels, class_labels):
    """Compute per-class and overall metrics. Returns dict with metrics and per_class rows."""
    total_tp = total_fp = total_tn = total_fn = 0
    all_precision, all_recall, all_f1, auc_scores = [], [], [], []
    per_class_rows = []

    for i, label in enumerate(class_labels):
        y_true = all_labels[:, i]
        y_pred = all_preds[:, i]
        y_prob = all_probs[:, i]

        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        support = tp + fn

        total_tp += tp
        total_fp += fp
        total_tn += tn
        total_fn += fn

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if support > 0:
            all_precision.append(precision)
            all_recall.append(recall)
            all_f1.append(f1)

        try:
            auc = roc_auc_score(y_true, y_prob)
            auc_scores.append(auc)
        except Exception:
            auc = 0.0

        per_class_rows.append((label, tp, fp, tn, fn, precision, recall, f1, auc, support))

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0

    macro_precision = np.mean(all_precision) if all_precision else 0
    macro_recall = np.mean(all_recall) if all_recall else 0
    macro_f1 = np.mean(all_f1) if all_f1 else 0
    macro_auc = np.mean(auc_scores) if auc_scores else 0

    return {
        "per_class_rows": per_class_rows,
        "total_tp": total_tp, "total_fp": total_fp, "total_tn": total_tn, "total_fn": total_fn,
        "micro_precision": micro_precision, "micro_recall": micro_recall, "micro_f1": micro_f1,
        "macro_precision": macro_precision, "macro_recall": macro_recall, "macro_f1": macro_f1, "macro_auc": macro_auc,
    }


REPORT_WIDTH = 80

def _section(title, width=REPORT_WIDTH):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def _get_labels(pred_row, class_labels):
    return [class_labels[j] for j in range(len(class_labels)) if pred_row[j]] or ["(none)"]

def print_evaluation_report(metrics, class_labels, all_preds, all_labels, n_samples=5):
    """Print per-class table, confusion matrix, and sample predictions."""
    per_class_rows = metrics["per_class_rows"]
    total_tp = metrics["total_tp"]
    total_fp = metrics["total_fp"]
    total_tn = metrics["total_tn"]
    total_fn = metrics["total_fn"]

    _section("PER-CLASS METRICS")
    print(f"{'Disease':<20} {'TP':>6} {'FP':>6} {'TN':>6} {'FN':>6} | {'Prec':>6} {'Recall':>6} {'F1':>6} {'AUC':>6} | {'Support':>7}")
    print("-" * REPORT_WIDTH)
    for row in per_class_rows:
        label, tp, fp, tn, fn, precision, recall, f1, auc, support = row
        print(f"{label:<20} {tp:>6} {fp:>6} {tn:>6} {fn:>6} | {precision:>6.3f} {recall:>6.3f} {f1:>6.3f} {auc:>6.3f} | {support:>7}")

    _section("OVERALL CONFUSION MATRIX (aggregated across all classes)")
    total_predictions = total_tp + total_fp + total_tn + total_fn
    pct = lambda x: 100 * x / total_predictions
    print(f"                    Predicted Positive    Predicted Negative")
    print(f"  Actual Positive      TP: {total_tp:>7} ({pct(total_tp):>5.2f}%)     FN: {total_fn:>7} ({pct(total_fn):>5.2f}%)")
    print(f"  Actual Negative      FP: {total_fp:>7} ({pct(total_fp):>5.2f}%)     TN: {total_tn:>7} ({pct(total_tn):>5.2f}%)")

    _section("SAMPLE PREDICTIONS (first 5)")
    for i in range(min(n_samples, len(all_preds))):
        pred_str = _get_labels(all_preds[i], class_labels)
        true_str = _get_labels(all_labels[i], class_labels)
        print(f"Image {i+1}:  Predicted: {pred_str}  |  Actual: {true_str}")


# Load Model 

In [22]:
# [Cell 23: Load Model]
# Run this cell when you want to load. Default: latest run. Or: load_model(run_id="abc123")
model, run_id = load_model()

Loading most recent training run: 38bac1c5673f45ed9359764f3d14ce5f


In [25]:
run_id

'38bac1c5673f45ed9359764f3d14ce5f'

# Model Evaluation

In [23]:
# [Cell 24: Comprehensive Model Evaluation]
# Run evaluation using helper functions from Cell 22
all_probs, all_preds, all_labels = run_inference(model, val_loader, device)
metrics = compute_per_class_metrics(all_probs, all_preds, all_labels, class_labels)
print_evaluation_report(metrics, class_labels, all_preds, all_labels)



PER-CLASS METRICS
Disease                  TP     FP     TN     FN |   Prec Recall     F1    AUC | Support
--------------------------------------------------------------------------------
Atelectasis               7     11  20067   2339 |  0.389  0.003  0.006  0.723 |    2346
Cardiomegaly              0      0  21897    527 |  0.000  0.000  0.000  0.727 |     527
Consolidation             0      0  21470    954 |  0.000  0.000  0.000  0.755 |     954
Edema                     0      0  21945    479 |  0.000  0.000  0.000  0.843 |     479
Effusion                630   1003  18836   1955 |  0.386  0.244  0.299  0.792 |    2585
Emphysema                 0      0  21909    515 |  0.000  0.000  0.000  0.809 |     515
Fibrosis                  0      0  22121    303 |  0.000  0.000  0.000  0.714 |     303
Hernia                    0      0  22386     38 |  0.000  0.000  0.000  0.744 |      38
Infiltration           1049   1833  16608   2934 |  0.364  0.263  0.306  0.665 |    3983
Mass      

# Log Metrics to MLflow

In [26]:
# [Cell 25: Log Evaluation Metrics to MLflow]
eval_metrics = {
    "eval_macro_auc": metrics["macro_auc"],
    "eval_macro_precision": metrics["macro_precision"],
    "eval_macro_recall": metrics["macro_recall"],
    "eval_macro_f1": metrics["macro_f1"],
    "eval_micro_precision": metrics["micro_precision"],
    "eval_micro_recall": metrics["micro_recall"],
    "eval_micro_f1": metrics["micro_f1"],
}

# MLflow: log evaluation metrics (with safety check to prevent duplicate logging)
if 'run_id' not in globals() or not run_id:
    print("No run_id available. Run the Load Model cell first.")
else:
    run_info = mlflow.get_run(run_id)
    already_evaluated = (
        "evaluation_complete" in run_info.data.tags
        or "eval_macro_auc" in run_info.data.metrics
    )
    if already_evaluated:
        print(f"Evaluation already logged for run {run_id}. Skipping to avoid duplicate metrics.")
    else:
        for metric_name, value in eval_metrics.items():
            mlflow.log_metric(metric_name, value, run_id=run_id)
        mlflow.MlflowClient().set_tag(run_id, "evaluation_complete", "true")
        print(f"Logged evaluation metrics to MLflow run {run_id}")

Evaluation already logged for run 38bac1c5673f45ed9359764f3d14ce5f. Skipping to avoid duplicate metrics.


In [27]:
# [Cell 26: Register Model Function]
def register_model(run_id=None, model_name=model_name):
    """Register model from MLflow run to registry. run_id=None uses latest (or mlflow_run_id from training). Returns ModelVersion."""
    if run_id:
        print(f"Registering specified run: {run_id}")
    elif "mlflow_run_id" in globals() and mlflow_run_id:
        run_id = mlflow_run_id
        print(f"Registering most recent training run: {run_id}")
    else:
        runs = mlflow.search_runs(experiment_names=["CXDD-Model"], order_by=["start_time DESC"], max_results=1)
        if runs.empty:
            raise RuntimeError("No runs found in CXDD-Model experiment. Train a model first.")
        run_id = runs.iloc[0]["run_id"]
        print(f"Registering latest run: {run_id}")

    run_info = mlflow.get_run(run_id)
    run_model = run_info.data.params.get("model")
    if run_model and run_model != model_name:
        import warnings
        warnings.warn(f"Run {run_id} was trained with model '{run_model}' but registering as '{model_name}'. Consider model_name='{run_model}'.")

    result = mlflow.register_model(model_uri=f"runs:/{run_id}/model", name=model_name)
    print(f"Registered as {model_name} version {result.version}")
    return result

In [28]:
# [Cell 27: Register Model]
# Run when you want to register. Default: latest run. Or: register_model(run_id="abc123")
model_version = register_model()

Registering most recent training run: 38bac1c5673f45ed9359764f3d14ce5f


2026/03/14 20:38:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/14 20:38:27 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
Successfully registered model 'ResNet50'.
2026/03/14 20:38:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 38bac1c5673f45ed9359764f3d14ce5f has no artifacts at artifact path 'model', registering model based on models:/m-341e1b954dda4454bbae292e04fba9e1 instead


Registered as ResNet50 version 1


Created version '1' of model 'ResNet50'.


# Diagnostics

In [3]:
# [Cell 28: Diagnostic - Check model output probabilities]

model.eval()
with torch.no_grad():
    # Get one batch
    images, labels = next(iter(val_loader))
    images = images.to(device)
    outputs = model(images)
    probs = torch.sigmoid(outputs)
    
    print("=" * 70)
    print("DIAGNOSTIC: Raw Model Outputs")
    print("=" * 70)
    
    # Statistics
    print(f"\nRaw outputs (before sigmoid):")
    print(f"  Min: {outputs.min().item():.4f}")
    print(f"  Max: {outputs.max().item():.4f}")
    print(f"  Mean: {outputs.mean().item():.4f}")
    
    print(f"\nProbabilities (after sigmoid):")
    print(f"  Min: {probs.min().item():.4f}")
    print(f"  Max: {probs.max().item():.4f}")
    print(f"  Mean: {probs.mean().item():.4f}")
    
    # Per-class max probabilities
    print(f"\nPer-class MAX probability across batch:")
    for i, label in enumerate(class_labels):
        max_prob = probs[:, i].max().item()
        print(f"  {label:<25}: {max_prob:.4f}")
    
    # Count predictions at different thresholds
    print(f"\nPredictions at different thresholds:")
    for thresh in [0.5, 0.3, 0.2, 0.1, 0.05]:
        count = (probs > thresh).sum().item()
        print(f"  Threshold {thresh}: {count} positive predictions")

NameError: name 'model' is not defined